In [1]:
from google.colab import files
uploaded = files.upload()

Saving Board_Game_Info.txt to Board_Game_Info.txt
Saving Bookings.txt to Bookings.txt
Saving feedbackManager.pyc to feedbackManager.pyc
Saving Game_Feedback.txt to Game_Feedback.txt
Saving menu.ipynb to menu.ipynb
Saving Rental.txt to Rental.txt
Saving Subscription_Info.txt to Subscription_Info.txt
Saving subscriptionManager.pyc to subscriptionManager.pyc
Saving Video_Game_Info.txt to Video_Game_Info.txt


# Helper Cell

**What This Cell Does:**

This cell sets up the helper functions and utilities for a game rental system. It provides shared functions that other cells use for:
* Customer validation and subscription checking
* Loading game data from CSV files
* Tracking rentals and availability
* Managing ratings and reviews
* Image handling from GitHub

**How It Should Be Used:**
* Run this cell FIRST before any other cells
* It creates reusable functions that other parts of the program depend on
* Does not display anything to the user - it's a "behind the scenes" setup cell
---

## **Imports Explained**

**IPython/Jupyter Display Libraries:**
* `clear_output` - Clears cell output (used for refreshing displays)
* `ipywidgets` - Creates interactive buttons, dropdowns, text boxes
* `display` - Shows widgets and content in the notebook
* `IPython` - Core Jupyter functionality

**Other Libraries**
* `threading` - Allows running code in parallel (for animations/timers)
* `time` - Time delays and timestamps
* `csv` - Reading/writing CSV files (customer data, rentals, games)
* `datetime` - Date handling for subscriptions and rental periods
* `os` - File system operations (checking if files exist)
* `matplotlib.pyplot` - Creating charts/graphs

**Caching Libraries**
* `requests` - Fetching images from URLs (GitHub)
* `BytesIO` - Handling image data in memory
---

## Global Variables:

**`IMAGE_CACHE = {}`**
* Stores downloaded images to avoid re-downloading

---

## Functions:

**`getCustomer(username)`**
* Looks up customer in subscription file and checks if subscription is active
* **`username`** - Customer ID to find

**`getLimit(customer)`**
* Returns rental limit based on subscription tier (Basic or Premium)
* **`customer`** - Customer data dictionary

**`get_user_rent_count(username)`**
* Counts how many games a user currently has rented
* **`username`** - Customer ID

**`get_game_rent_count(game_id)`**
* Counts total times a game has been rented
* **`game_id`** - Game's unique ID

**`get_average_rating(game_id)`**
* Calculates average star rating from all feedback
* **`game_id`** - Game's unique ID

**`get_game_reviews(game_id)`**
* Gets all reviews (ratings + comments) for a specific game
* **`game_id`** - Game's unique ID

**`load_games(filename, game_type)`**
* Loads games from CSV file and adds rating/rental data
* **`filename`** - Path to CSV file
* **`game_type`** - Type label (e.g., "Board" or "Video")

**`load_rentals()`**
* Gets all currently active rentals (games not yet returned)

**`create_star_display(rating, string)`**
* Creates visual star rating display (★★★½☆)
* **`rating`** - Numeric rating value
* **`string`** - If None, returns widget; otherwise returns string

**`get_cached_image(image_url)`**
* Downloads and caches images from GitHub to avoid re-downloading
* **`image_url`** - URL to the image

**`get_image_format(game, image)`**
* Detects image file format from URL
* **`game`** - Game dictionary containing image URL
* **`image`** - Image data

**`get_game_rent_count_by_period(game_id, days=None)`**
* Counts the number of times a game has been rented, optionally filtered by a time period (e.g., last 30 days)
* **`game_id`** - Game's unique ID to count rentals for
* **`days`** - Optional number of days to look back (if None, counts all rentals)

In [2]:
from IPython.core.display import clear_output
import ipywidgets as widgets
from IPython.display import display
import IPython
import threading
import time
import feedbackManager as fmDS
import subscriptionManager as smDS
import csv
from datetime import datetime, timedelta
import os
import matplotlib.pyplot as plt

# These two are libraries necessary for fetching the images from my code
# from a github repository, since I cannot store images in the files
import requests
from io import BytesIO

# =================
# Global Variables
# =================

IMAGE_CACHE = {}

# ============================================
# HELPER FUNCTIONS
# ============================================

# Shared utilities for customer validation, game loading, ratings, and rental tracking

def getCustomer(username):
  today = datetime.today().strftime('%Y-%m-%d')
  # Look up customer in subscription file
  with open("Subscription_Info.txt", "r") as file:
    reader = csv.DictReader(file)
    for row in reader:
      if row["CustomerID"] == username:
        if row["EndDate"] < today:
          return -1
        return row
    return None


def getLimit(customer):
  # Returns rental limit based on subscription tier
  if customer["SubscriptionType"] == 'Basic':
    return smDS.BASIC_LIMIT
  else:
    return smDS.PREMIUM_LIMIT


def get_user_rent_count(username):
  # Count active rentals for a user (no return date)
  count = 0
  if not os.path.exists("Rental.txt"):
    return count
  with open("Rental.txt", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
      if row["CustomerID"] == username and row['ReturnDate'] == "":
        count += 1
  return count


def get_game_rent_count(game_id):
  # Total times a game has been rented
  count = 0
  if not os.path.exists("Rental.txt"):
    return count
  with open("Rental.txt", 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
      if row["GameID"] == game_id:
        count += 1
  return count


def get_average_rating(game_id):
  # Calculate average star rating from feedback
  if not os.path.exists("Game_Feedback.txt"):
    return None
  ratings = []
  with open("Game_Feedback.txt", 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
      if row["GameID"] == game_id:
        ratings.append(int(row["Rating"]))
  if not ratings:
    return None
  return sum(ratings) / len(ratings)


def get_game_reviews(game_id):
  # Pull all reviews for a specific game
  all_feedback = fmDS.load_feedback('Game_Feedback.txt')
  reviews = []
  for feedback in all_feedback:
    if feedback["GameID"] == game_id:
      reviews.append({
        "Rating": int(feedback["Rating"]),
        "Comment": feedback["Comments"]
    })
  return reviews


def load_games(filename, game_type):
  # Load games from CSV and attach rating/rental data
  games = []
  with open(filename, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
      games.append({
        "GameID": row["GameID"],
        "Name": row["Name"],
        "Image": row["Image"],
        "NoPlayers": row["NoPlayers"],
        "Genre": row["Genre"],
        "PurchaseDate": row["PurchaseDate"],
        "Type": game_type,
        "Available": True,
        "Rating": get_average_rating(row["GameID"]),
        "RentalCount": get_game_rent_count(row["GameID"])
      })
  return games


def load_rentals():
  # Get all currently active rentals (no return date set)
  rentals = {}
  if os.path.exists("Rental.txt"):
    with open("Rental.txt", 'r') as f:
      reader = csv.DictReader(f)
      for row in reader:
        if row["ReturnDate"] == "":
          rentals[row["GameID"]] = row
  return rentals


def create_star_display(rating,string):
  # Creates star rating widget (★★★½☆ format)
  if rating == None:
    return widgets.Label(value="No ratings yet", layout=widgets.Layout(height='25px'))

  # Round to nearest half star
  rounded_rating = round(rating * 2) / 2
  full_stars = int(rounded_rating)
  half_star = (rounded_rating % 1) != 0
  empty_stars = 5 - full_stars - (1 if half_star else 0)

  star_str = "★" * full_stars
  if half_star:
    star_str += "½"
  star_str += "☆" * empty_stars

  if string == None:

    display_text = f"{star_str} {rating:.1f}"

    return widgets.HTML(
      value=f"<div style='color: #FFA500; font-size: 16px;'>{display_text}</div>",
      layout=widgets.Layout(height="25px")
    )
  else:
    return star_str

def get_cached_image(image_url):
  # Fetches an image from github and "caches" it

  if image_url in IMAGE_CACHE:
    return  IMAGE_CACHE[image_url]

  try:
    response = requests.get(image_url, timeout=10)
    response.raise_for_status()
    image = response.content
    IMAGE_CACHE[image_url] = image
    return image

  except Exception as e:
    print(f"Error fetching image from URL: {image_url}")
    print(e)
    return None

def get_image_format(game,image):
  if image:
  # Detect format from URL or default to common formats
    image_url = game["Image"].lower()
    if '.png' in image_url:
      img_format = 'png'
      return img_format
    elif '.jpg' in image_url or '.jpeg' in image_url:
      img_format = 'jpeg'
      return img_format
    elif '.gif' in image_url:
      img_format = 'gif'
      return img_format
    elif '.avif' in image_url:
      img_format = 'avif'
      return img_format
    else:
      img_format = 'png'
      return img_format

def get_game_rent_count_by_period(game_id, days=None):
  # Finds the number of times a game has been rented based on a period
  count = 0
  if not os.path.exists("Rental.txt"):
    return count

  # Calculate cutoff date if days specified
  cutoff_date = None
  if days is not None:
    cutoff_date = (datetime.today() - timedelta(days=days)).strftime('%Y-%m-%d')

  with open("Rental.txt", 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
      if row["GameID"] == game_id:
        # If there is a time filter, check the rental date
        if cutoff_date and row["RentalDate"] < cutoff_date:
          continue
        count += 1
    return count

# Game Card Display Cell

**What This Cell Does:**

This cell creates the visual display cards for each game in the rental system. Each card shows:
* A Game image, fetched -
https://github.com/Visheon/Coursework-Images---Game-Store
* Game information (name, players, genre)
* Star rating (if available)
* Availability status (Available/Rented)
* Action buttons (Rent/Return and Reviews)

**How It Should Be Used:**
* Run this cell after the Helper Cell
* Provides the visual game card component used in the dashboard
* Does not display anything by itself - creates cards when other cells call this function using the status of current games in rental.txt to decide the ui to display

---

## Functions:

**`create_game_card(game, rent_callback, return_callback, reviews_callback)`**
* Creates a visual card widget for displaying a single game with all its details and action buttons
* **`game`** - Game dictionary containing all game information
* **`rent_callback`** - Function to call when Rent button is clicked
* **`return_callback`** - Function to call when Return button is clicked
* **`reviews_callback`** - Function to call when Reviews button is clicked

In [3]:
# ============================================
# GAME CARD DISPLAY
# ============================================

# Creates the visual card for each game with image, info, rating, and action buttons

def create_game_card(game, rent_callback, return_callback, reviews_callback):

  image = get_cached_image(game["Image"])
  img_format = get_image_format(game,image)

  # Load game image or show a placeholder if there is no image
  try:
    img_widget = widgets.Image(
      value=image,
      format=img_format,
      layout=widgets.Layout(max_width="280px",
                            max_height="200px",
                            min_height="200px",
                            width="100%",
                            height="auto",
                            object_fit="contain"))
  except:
    img_widget = widgets.Label(value="No Image")

  # Main action button (RENT or RETURN)
  rent_button = widgets.Button(
    description="RENT" if game["Available"] else "RETURN",
    layout=widgets.Layout(width="120px", height="40px")
  )

  if game["Available"]:
    rent_button.style.button_color = "#0da602"
    rent_button.on_click(lambda x: rent_callback(game))
  else:
    rent_button.style.button_color = "darkgray"
    rent_button.on_click(lambda x: return_callback(game))

  # Reviews button
  reviews_button = widgets.Button(
    description="📝 Reviews",
    layout=widgets.Layout(width="120px", height="35px"),
    button_style='info'
  )
  reviews_button.on_click(lambda x: reviews_callback(game))

  # Basic game info
  title = widgets.Label(value=game['Name'])
  players = widgets.Label(value=f"Players: {game['NoPlayers']}")
  genre = widgets.Label(value=f"Genre: {game['Genre']}")

  # Build status section (available vs rented)
  if game['Available']:
    border_color = "green"
    rating_widget = create_star_display(game['Rating'],None)
    status = widgets.HTML(
      value="<span style='color: green; font-weight: bold;'>AVAILABLE</span>",
    )
    card_children = [title, img_widget, players, genre, rating_widget, status, rent_button, reviews_button]
  else:
    border_color = "red"
    # Show who rented it if that info exists
    if "RentalInfo" in game:
      rental = game["RentalInfo"]
      status = widgets.HTML(
        value=f"<div style='text-align: center;'>"
        f"<span style='color: red; font-weight: bold;'>RENTED</span><br>"
        f"Rented by: {rental['CustomerID']}<br>",
      )
    else:
      status = widgets.HTML(
        value="<span style='color: red; font-weight: bold;'>RENTED</span>",
      )
    card_children = [title, img_widget, players, genre, status, rent_button, reviews_button]

  # Assemble card with colored border
  card = widgets.VBox(
    card_children,
    layout=widgets.Layout(
      border=f"2px solid {border_color}",
      padding="4px",
      margin="6px",
      width="280px",
      max_width="400px",
      min_width="200px",
      min_height="500px",
      align_items="center",
      grid_gap="2px"
    )
  )

  return card

# Game Rental Cell

**What This Cell Does:**

Once the manager has chosen which game is to be rented from the game cards, this cell handles the game rental process. It:
* Takes in and validates the customer's username and subscription status
* Checks if the customer has reached their rental limit, either the Basic or Premium tier based on their account
* Creates a rental record in the Rental.txt file, leaving an empty space on the return date
* Displays a popup interface for entering customer ID and confirming the rental

Since the manager has selected a game card from the UI, there is no need for them to enter a gameID as the game card already has the ID in the game variable as a parameter

**How It Should Be Used:**
* Run this cell after the Helper Cell and Game Card Display Cell
* Called automatically when a user clicks the "RENT" button on a game card
* Displays a popup asking for Customer ID, then processes the rental

---

## Functions:

**`rent_game_on_confirm(btn, user_input, message_label, return_to_callback, game)`**
* Processes the rental after user confirms - validates customer, checks limits, and saves rental record
* **`btn`** - Button widget that triggered the event
* **`user_input`** - Text widget containing the Customer ID
* **`message_label`** - Label widget to display success/error messages
* **`return_to_callback`** - Function to call to return to main screen
* **`game`** - Game dictionary being rented

**`rent_game_on_cancel(btn, cancel_callback)`**
* Cancels the rental process and returns to previous screen
* **`btn`** - Button widget that triggered the event
* **`cancel_callback`** - Function to call when cancelling

**`rent_game(game, return_to_callback, cancel_callback)`**
* Creates and displays the rental popup interface with Customer ID input and Confirm/Cancel buttons
* **`game`** - Game dictionary being rented
* **`return_to_callback`** - Function to call after successful rental
* **`cancel_callback`** - Function to call if rental is cancelled

In [4]:
# ============================================
# GAME RENTAL
# ============================================

# Handles the rental process with customer validation and checks the subscription limit.

def rent_game_on_confirm(btn, user_input, message_label, return_to_callback, game):
  username = user_input.value.strip()
  customer = getCustomer(username)

  # Validate customer exists
  if customer == None:
    message_label.value = "❌ Invalid Username"
    return
  if customer == -1:
    message_label.value = "❌ Subscription has expired"
    return

  # Check if user has hit their rental limit
  limit = getLimit(customer)
  current_rented = get_user_rent_count(username)

  if current_rented >= limit:
    message_label.value = f"❌ You have reached your rental limit ({limit}). Please return a game first."
    return

  # Create rental record
  rental_entry = {
    "GameID": game["GameID"],
    "RentalDate": datetime.now().strftime("%Y-%m-%d"),
    "ReturnDate": "",  # Empty until returned
    "CustomerID": username
  }

  # Append to rental file
  file_exists = os.path.exists("Rental.txt")
  with open("Rental.txt", "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["GameID", "RentalDate", "ReturnDate", "CustomerID"])
    if not file_exists:
      writer.writeheader()
    writer.writerow(rental_entry)

  message_label.value = "✔ Game rented successfully!"
  time.sleep(2)
  clear_output(wait=True)
  return_to_callback()


def rent_game_on_cancel(btn, cancel_callback):
  clear_output(wait=True)
  cancel_callback()


def rent_game(game, return_to_callback, cancel_callback):
  user_input = widgets.Text(description="Customer ID:")
  message_label = widgets.Label(value="")
  confirm_button = widgets.Button(description="Confirm", button_style="success")
  cancel_button = widgets.Button(description="Cancel", button_style="danger")

  confirm_button.on_click(lambda btn: rent_game_on_confirm(btn, user_input, message_label, return_to_callback, game))
  cancel_button.on_click(lambda btn: rent_game_on_cancel(btn, cancel_callback))

  prompt_label = widgets.Label(value="Enter Customer ID to rent this game:")
  popup = widgets.VBox(
    [prompt_label, user_input, message_label, widgets.HBox([confirm_button, cancel_button])],
    layout=widgets.Layout(align_items='center', padding='10px')
  )

  clear_output(wait=True)
  display(popup)

# Game Return Cell

**What This Cell Does:**

This cell handles the game return process with mandatory star rating and optional comments collection. It:
* Displays a popup with an interactive star rating (1-5 stars)
* Collects written comments from the user
* Saves feedback to Game_Feedback.txt file
* Updates the rental record with the return date
* Marks the game as available again

**How It Should Be Used:**
* Run this cell after the Helper Cell
* Called automatically when a user clicks the "RETURN" button on a rented game card
* Requires the user to provide a star rating before submitting where the comments are optional

---

## Functions:

**`return_game_on_star_click(b, rating, selected_rating, star_buttons)`**
* Handles star button clicks and updates the visual star display to show selected rating
* **`b`** - Button widget that was clicked
* **`rating`** - The star rating value (1-5) that was clicked
* **`selected_rating`** - List containing the currently selected rating
* **`star_buttons`** - List of all star button widgets

**`return_game_create_star_button(rating, selected_rating, star_buttons)`**
* Creates a single clickable star button for the rating interface
* **`rating`** - The rating value (1-5) this button represents
* **`selected_rating`** - Same as above
* **`star_buttons`** - Same as above

**`return_game_on_submit(btn, selected_rating, comment_input, message_label, game, return_to_callback)`**
* Processes the return after user submits - it validates the rating, saves feedback, and updates the rental record
* **`btn`** - Same as above
* **`selected_rating`** - Same as above
* **`comment_input`** - Textarea widget containing user's comments
* **`message_label`** - Label widget to display error messages
* **`game`** - Game dictionary being returned
* **`return_to_callback`** - Function that reloads the dashboard to show updated game availability

**`return_game_on_cancel(btn, cancel_callback)`**
* Cancels the return process and returns to the previous screen
* **`btn`** - Same as above
* **`cancel_callback`** - Function that returns to previous screen without reloading

**`return_game(game, return_to_callback, cancel_callback)`**
* Creates and displays the return popup interface with star rating, comment box, and Submit/Cancel buttons
* **`game`** - Same as above
* **`return_to_callback`** - Same as above (reloads dashboard)
* **`cancel_callback`** - Same as above (returns without reloading)

In [5]:
# ============================================
# GAME RETURN
# ============================================

# Handles returns with mandatory feedback collection (star rating + comments)

def return_game_on_star_click(btn, rating, selected_rating, star_buttons):
  selected_rating[0] = rating
  # Fill stars up to selected rating
  for i, star_btn in enumerate(star_buttons):
    if i < rating:
      star_btn.button_style = 'warning'
    else:
      star_btn.button_style = ''


def return_game_create_star_button(rating, selected_rating, star_buttons):
  btn = widgets.Button(
    description='★',
    layout=widgets.Layout(width='50px', height='50px'),
    style={'font_size': '24px'}
  )
  btn.on_click(lambda btn: return_game_on_star_click(btn, rating, selected_rating, star_buttons))
  return btn


def return_game_on_submit(btn, selected_rating, comment_input, message_label, game, return_to_callback):
  # Validate feedback before processing return
  if selected_rating[0] == 0:
    message_label.value = "❌ Please select a rating"
    return

  comment_text = comment_input.value.strip()
  if not comment_text:
    comment_text = "No comments given."

  # Save feedback
  fmDS.add_feedback(
    game['GameID'],
    selected_rating[0],
    comment_text,
    'Game_Feedback.txt'
  )

  # Update rental record with return date
  updated_rows = []
  with open("Rental.txt", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
      if row["GameID"] == game["GameID"] and row["ReturnDate"] == "":
        row["ReturnDate"] = datetime.now().strftime("%Y-%m-%d")
      updated_rows.append(row)

  # Write back all rentals
  with open("Rental.txt", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["GameID", "RentalDate", "ReturnDate", "CustomerID"])
    writer.writeheader()
    writer.writerows(updated_rows)

  game["Available"] = True

  clear_output(wait=True)
  success_label = widgets.Label(value="✔ Game returned and feedback submitted successfully!")
  display(success_label)
  time.sleep(2)
  return_to_callback()


def return_game_on_cancel(btn, cancel_callback):
  clear_output(wait=True)
  cancel_callback()


def return_game(game, return_to_callback, cancel_callback):
  if game["Available"]:
    return

  star_buttons = []
  selected_rating = [0]  # Use list to maintain reference in nested function

  # Create star buttons
  for i in range(1, 6):
    star_buttons.append(return_game_create_star_button(i, selected_rating, star_buttons))

  stars_container = widgets.HBox(
    star_buttons,
    layout=widgets.Layout(justify_content='center', margin='10px 0')
  )

  # Comment input
  comment_label = widgets.Label(
    value='Comments:',
    layout=widgets.Layout(width='400px', margin='10px 0 5px 0')
  )
  comment_input = widgets.Textarea(
    placeholder='Share your experience with this game...',
    layout=widgets.Layout(width='400px', height='100px')
  )

  submit_button = widgets.Button(
    description="Submit Feedback",
    button_style="success",
    layout=widgets.Layout(width='150px')
  )
  cancel_button = widgets.Button(
    description="Cancel",
    button_style="danger",
    layout=widgets.Layout(width='150px')
  )

  message_label = widgets.Label(value="")

  submit_button.on_click(lambda btn: return_game_on_submit(btn, selected_rating, comment_input, message_label, game, return_to_callback))
  cancel_button.on_click(lambda btn: return_game_on_cancel(btn, cancel_callback))

  title_label = widgets.Label(
    value=f"Return Game: {game['Name']}",
    layout=widgets.Layout(margin='0 0 10px 0')
  )
  rating_label = widgets.Label(
    value="Rate your experience (click stars):",
    layout=widgets.Layout(margin='10px 0 5px 0')
  )

  popup = widgets.VBox(
    [
      title_label,
      rating_label,
      stars_container,
      comment_label,
      comment_input,
      message_label,
      widgets.HBox(
        [submit_button, cancel_button],
          layout=widgets.Layout(justify_content='center', margin='10px 0')
      )
    ],
      layout=widgets.Layout(
        align_items='center',
        padding='20px',
        border='2px solid gray',
        width='500px',
        margin='50px auto'
      )
  )

  clear_output(wait=True)
  display(popup)

# Reviews Display Cell

**What This Cell Does:**

This cell displays all reviews for a selected game. It:
* Shows the average star rating at the top with total review count
* Lists all individual reviews with their star ratings and comments
* Provides a scrollable container if there are many reviews
* Displays "No reviews yet" if the game has no feedback

**How It Should Be Used:**
* Run this cell after the Helper Cell
* It is called automatically when a user clicks the Reviews button on any game card
* Displays a popup that shows all reviews and can be closed to return to the previous screen

---

## Functions:

**`show_reviews_on_close(btn, cancel_callback)`**
* Closes the reviews popup and returns to the previous screen
* **`btn`** - Button widget that was clicked
* **`cancel_callback`** - Function that returns to previous screen without reloading

**`show_reviews(game, return_to_callback, cancel_callback)`**
* Creates and displays the reviews popup with average rating, all individual reviews, and a close button
* **`game`** - Game dictionary to show reviews for
* **`return_to_callback`** - Function that reloads the dashboard (not used in this function)
* **`cancel_callback`** - Same as above (returns without reloading)

In [6]:
# ============================================
# REVIEWS DISPLAY
# ============================================

# Shows all reviews for a game with average rating

def show_reviews_on_close(btn, cancel_callback):
  clear_output(wait=True)
  cancel_callback()


def show_reviews(game, return_to_callback, cancel_callback):
  reviews = get_game_reviews(game["GameID"])

  title = widgets.HTML(
    value=f"<h3 style='margin: 0 0 10px 0;'>Reviews for {game['Name']}</h3>"
  )

  # Show average rating at top
  avg_rating = game.get('Rating')
  if avg_rating:
    avg_display = widgets.HTML(
    value=f"<div style='text-align: center; margin: 10px 0;'>"
          f"<span style='color: #FFA500; font-size: 20px;'>{'★' * int(round(avg_rating))}{'☆' * (5 - int(round(avg_rating)))}</span><br>"
          f"<span style='font-size: 16px;'>Average: {avg_rating:.1f}/5.0 ({len(reviews)} reviews)</span>"
          f"</div>"
    )
  else:
    avg_display = widgets.Label(value="No reviews yet")

  close_btn = widgets.Button(
    description="Close",
    button_style='primary',
    layout=widgets.Layout(width='150px', margin='10px 0')
  )

  close_btn.on_click(lambda btn: show_reviews_on_close(btn, cancel_callback))

  # Build review cards
  if not reviews:
    review_widgets = [widgets.Label(
      value="No reviews yet.",
      layout=widgets.Layout(margin='20px 0')
  )]
  else:
    review_widgets = []
    for review in reviews:
      stars = create_star_display(review["Rating"],None)
      comment = widgets.Label(
        value=review["Comment"],
        layout=widgets.Layout(margin='5px 0 0 0')
      )

      review_card = widgets.VBox(
        [stars, comment],
        layout=widgets.Layout(
          border='1px solid #ccc',
          padding='10px',
          margin='5px 0',
          width='100%',
          background_color='#f9f9f9'
        )
      )
      review_widgets.append(review_card)

  # Scrollable container for reviews
  reviews_container = widgets.VBox(
    review_widgets,
    layout=widgets.Layout(
      max_height='400px',
      overflow_y='auto',
      width='100%',
      padding='10px'
    )
  )

  popup = widgets.VBox(
    [title, avg_display, close_btn, reviews_container],
    layout=widgets.Layout(
      width='500px',
      max_height='600px',
      padding='20px',
      border='2px solid gray',
      margin='20px auto',
      align_items='center'
    )
  )

  clear_output(wait=True)
  display(popup)

# Bookings Cell

**What This Cell Does:**

This cell manages in-store gaming session bookings with capacity management. It:
* Allows customers to book gaming sessions for next 30 days
* Offers two time slots per day (2pm-6pm and 6pm-10pm)
* Enforces a 50-person capacity limit per time slot
* Limits party size to 1-3 guests per booking (using a slider)
* Prevents duplicate bookings (one active booking per customer)
* Automatically removes expired bookings (past dates)
* Displays all active bookings in chronological order (sorted by date, then time)
* Shows capacity information for selected dates (bookings/slots remaining)
* Provides date filtering to view bookings for specific dates
* Displays bookings in a 2-column grid layout with cancel option

**How It Should Be Used:**
* Run this cell after the Helper Cell
* Creates the bookings interface that appears in the main dashboard
* Allows users to book sessions, view existing bookings filtered by date, see capacity info, and cancel bookings
* Expired bookings are automatically cleaned up when the view is created

---

## Functions:

**`book_session_check_capacity(date_str, time_slot)`**
* Calculates the total number of guests already booked for a specific date and time slot
* **`date_str`** - Date in YYYY-MM-DD format
* **`time_slot`** - Time slot string (e.g., "14:00-18:00 (2pm-6pm)")

**`bookings_get_capacity_for_date(date_str)`**
* Gets capacity information (current bookings) for all time slots on a specific date
* **`date_str`** - Same as above

**`bookings_remove_expired()`**
* Removes bookings that have passed their date by comparing booking dates to today's date
* **No parameters**

**`book_session_on_confirm(btn, user_input, message, booking_date, time_dropdown, party_size, return_to_callback)`**
* Processes the booking after user confirms - validates customer, checks for duplicates, verifies capacity, and saves booking
* **`btn`** - Button widget that triggered the event
* **`user_input`** - Text widget containing the User ID
* **`message`** - Label widget to display error/success messages
* **`booking_date`** - Dropdown widget with selected date
* **`time_dropdown`** - Dropdown widget with selected time slot
* **`party_size`** - IntSlider widget with number of guests (1-3)
* **`return_to_callback`** - Function that reloads the dashboard

**`book_session_on_cancel(btn, cancel_callback)`**
* Cancels the booking process and returns to previous screen
* **`btn`** - Same as above
* **`cancel_callback`** - Function that returns to previous screen without reloading

**`book_session(return_to_callback, cancel_callback)`**
* Creates and displays the booking form popup with User ID, date, time, party size slider inputs and Confirm/Cancel buttons
* **`return_to_callback`** - Same as above (reloads dashboard)
* **`cancel_callback`** - Same as above (returns without reloading)

**`bookings_cancel_booking(btn, user_id, return_to_callback)`**
* Removes a booking from the Bookings.txt file and refreshes the view
* **`btn`** - Same as above
* **`user_id`** - User ID of the booking to cancel
* **`return_to_callback`** - Same as above (reloads dashboard)

**`bookings_update_capacity_display(info_box_html, date_filter)`**
* Updates the capacity information display box to show bookings and remaining slots for the selected date
* **`info_box_html`** - HTML widget displaying the capacity information box
* **`date_filter`** - Dropdown widget with selected date filter

**`bookings_filter_by_date(change, date_filter, booking_cards_container, info_box_html, all_bookings, return_to_callback)`**
* Filters bookings by selected date, updates capacity display, and rebuilds the booking cards grid
* **`change`** - Event object from dropdown change
* **`date_filter`** - Same as above
* **`booking_cards_container`** - VBox widget where filtered booking cards are displayed
* **`info_box_html`** - Same as above
* **`all_bookings`** - List of all booking dictionaries (sorted chronologically)
* **`return_to_callback`** - Same as above (reloads dashboard)

**`create_bookings_view(book_callback, return_to_callback)`**
* Creates the main bookings tab view with capacity info box, date filter, "Book New Session" button, and booking cards grid (2 columns)
* **`book_callback`** - Function to call when "Book New Session" button is clicked
* **`return_to_callback`** - Same as above (reloads dashboard)

In [24]:
# ============================================
# BOOKINGS
# ============================================

# In-store gaming session bookings with capacity management (50 person limit per time slot)

def book_session_check_capacity(date_str, time_slot):
  # Calculate current bookings for a time slot
  total_guests = 0
  if os.path.exists("Bookings.txt") and os.path.getsize("Bookings.txt") > 0:
    with open("Bookings.txt", 'r') as f:
      reader = csv.DictReader(f)
      for row in reader:
        if row["BookingDate"] == date_str and row["Time"] == time_slot:
          total_guests += int(row["NoGuests"])
  return total_guests


def bookings_get_capacity_for_date(date_str):
  # Get capacity information for all time slots on a specific date
  capacity_info = {
    "14:00-18:00 (2pm-6pm)": 0,
    "18:00-22:00 (6pm-10pm)": 0
  }

  if os.path.exists("Bookings.txt") and os.path.getsize("Bookings.txt") > 0:
    with open("Bookings.txt", 'r') as f:
      reader = csv.DictReader(f)
      for row in reader:
        if row["BookingDate"] == date_str and row["Time"] in capacity_info:
          capacity_info[row["Time"]] += int(row["NoGuests"])

  return capacity_info


def bookings_remove_expired():
  # Remove bookings that have passed their date
  if not os.path.exists("Bookings.txt") or os.path.getsize("Bookings.txt") == 0:
    return

  today = datetime.now().date()
  updated_rows = []

  with open("Bookings.txt", "r") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    for row in reader:
      booking_date = datetime.strptime(row["BookingDate"], "%Y-%m-%d").date()
      # Keep only future bookings
      if booking_date >= today:
        updated_rows.append(row)

  # Write back only active bookings
  with open("Bookings.txt", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(updated_rows)


def book_session_on_confirm(btn, user_input, message, booking_date, time_dropdown, party_size, return_to_callback):
  username = user_input.value.strip()
  customer = getCustomer(username)

  if customer == None:
    message.value = "❌ Invalid User ID"
    return
  elif customer == -1:
    message.value = "❌ Subscription has expired"
    return

  if booking_date.value == None:
    message.value = "❌ Please select a date"
    return

  if party_size.value < 1 or party_size.value > 3:
      message.value = "❌ Party size must be between 1 and 3 guests"
      return

  # Check for duplicate bookings
  if os.path.exists("Bookings.txt"):
    with open("Bookings.txt", 'r') as f:
      reader = csv.DictReader(f)
      for row in reader:
        if row["UserId"] == username and booking_date.value == row["BookingDate"]:
          message.value = "❌ User already has an active booking. Please cancel it first."
          return

        if (row["BookingDate"] == booking_date.value and
          row["Time"] == time_dropdown.value):
          message.value = "❌ This time slot is already fully booked. Please choose another time."
          return

  # Check capacity limit (50 people per slot)
  current_capacity = book_session_check_capacity(booking_date.value, time_dropdown.value)

  if current_capacity + party_size.value > 50:
    message.value = f"❌ Not enough space! Only {50 - current_capacity} spots left for this time slot."
    return

  # Create booking
  booking_entry = {
    "UserId": username,
    "BookingDate": booking_date.value,
    "Time": time_dropdown.value,
    "NoGuests": str(party_size.value)
  }

  file_exists = os.path.exists("Bookings.txt") and os.path.getsize("Bookings.txt") > 0
  with open("Bookings.txt", "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["UserId", "BookingDate", "Time", "NoGuests"])
    if not file_exists:
      writer.writeheader()
    writer.writerow(booking_entry)

  message.value = "✔ Session booked successfully!"
  time.sleep(2)
  clear_output(wait=True)
  return_to_callback()


def book_session_on_cancel(btn, cancel_callback):
  clear_output(wait=True)
  cancel_callback()


def book_session(return_to_callback, cancel_callback):
  popup = widgets.VBox()
  user_input = widgets.Text(description="User ID:", style={'description_width': '100px'})

  # Generate next 30 days of dates
  today = datetime.now().date()
  date_options = [(today + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(1, 31)]
  booking_date = widgets.Dropdown(
    options=date_options,
    description="Date:",
    style={'description_width': '100px'}
  )

  times = ["14:00-18:00 (2pm-6pm)", "18:00-22:00 (6pm-10pm)"]
  time_dropdown = widgets.Dropdown(
    options=times,
    description="Time:",
    style={'description_width': '100px'}
  )

  party_size = widgets.IntSlider(
    description="No. Guests:",
    value=1,
    min=1,
    max=3,
    step=1,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='300px')
  )

  confirm_button = widgets.Button(description="Book Session", button_style="success")
  cancel_button = widgets.Button(description="Cancel", button_style="danger")
  message = widgets.Label(value="")

  confirm_button.on_click(lambda btn: book_session_on_confirm(btn, user_input, message, booking_date, time_dropdown, party_size, return_to_callback))
  cancel_button.on_click(lambda btn: book_session_on_cancel(btn, cancel_callback))

  title_label = widgets.Label(value="Book an In-Store Gaming Session")
  popup = widgets.VBox(
    [title_label, user_input, booking_date, time_dropdown, party_size, message,
      widgets.HBox([confirm_button, cancel_button], layout=widgets.Layout(justify_content='center'))],
      layout=widgets.Layout(align_items='center', padding='10px'))

  clear_output(wait=True)
  display(popup)


def bookings_cancel_booking(btn, user_id, booking_date, booking_time, return_to_callback):
  # Removes a specific booking from file based on UserId, Date, and Time
  updated_rows = []
  with open("Bookings.txt", "r") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    for r in reader:
      # Keep all bookings except the one that matches all three criteria
      if not (r["UserId"] == user_id and r["BookingDate"] == booking_date and r["Time"] == booking_time):
        updated_rows.append(r)

  with open("Bookings.txt", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(updated_rows)

  return_to_callback()


def bookings_update_capacity_display(info_box_html, date_filter):
  # Update the entire HTML info box based on selected date"""
  selected_date = date_filter.value

  if selected_date == "All Dates":
    info_box_html.value = """
    <div style="
      background-color: #2E7D32;
      padding: 20px;
      border-radius: 8px;
      width: 750px;
      margin: 0 auto;
    ">
    <div style="
      display: flex;
      justify-content: space-between;
      align-items: center;
      margin-bottom: 20px;
    ">
      <span style="font-weight: bold; font-size: 16px; color: white;">
        📋 In-Store Session Bookings
      </span>
      </div>
      <div style="color: white; line-height: 1.8;">
      <div style="font-weight: bold;">Select a date to view capacity</div>
      </div>
      </div>
      """
    return

  # Get capacity for the selected date
  capacity_info = bookings_get_capacity_for_date(selected_date)

  # Date Formatting
  date_obj = datetime.strptime(selected_date, "%Y-%m-%d")
  day_name = date_obj.strftime("%A")
  formatted_date = date_obj.strftime("%B %d, %Y")

  today = datetime.now().date()
  is_today = date_obj.date() == today
  today_badge = " TODAY" if is_today else ""

  # Calculate capacity info
  afternoon_booked = capacity_info["14:00-18:00 (2pm-6pm)"]
  afternoon_remaining = 50 - afternoon_booked
  evening_booked = capacity_info["18:00-22:00 (6pm-10pm)"]
  evening_remaining = 50 - evening_booked

  # Update the entire HTML box with capacity information
  info_box_html.value = f"""
  <div style="
    background-color: #2E7D32;
    padding: 20px;
    border-radius: 8px;
    width: 750px;
    margin: 0 auto;
  ">
  <div style="
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 20px;
  ">
    <span style="font-weight: bold; font-size: 16px; color: white;">
    📋 In-Store Session Bookings
    </span>
    </div>
    <div style="color: white; line-height: 1.8;">
      <div style="font-weight: bold; margin-bottom: 10px;">📅 {day_name}, {formatted_date}{today_badge}</div>
      <div>2pm-6pm: {afternoon_booked}/50 booked ({afternoon_remaining} remaining)</div>
      <div>6pm-10pm: {evening_booked}/50 booked ({evening_remaining} remaining)</div>
    </div>
    </div>
    """


def bookings_filter_by_date(change, date_filter, booking_cards_container, info_box_html, all_bookings, return_to_callback):
  # Update capacity display in the header
  bookings_update_capacity_display(info_box_html, date_filter)

  # Filter bookings based on selected date
  selected_date = date_filter.value

  if selected_date == "All Dates":
    filtered_bookings = all_bookings
  else:
    filtered_bookings = [b for b in all_bookings if b["BookingDate"] == selected_date]

  # Rebuild the booking cards with filtered data
  if not filtered_bookings:
    no_bookings = widgets.Label(
      value=f"No bookings for {selected_date}" if selected_date != "All Dates" else "No active bookings",
      layout=widgets.Layout(margin='50px 0')
    )
    no_bookings_centered = widgets.HBox(
      [no_bookings],
      layout=widgets.Layout(justify_content='center')
    )
    booking_cards_container.children = [no_bookings_centered]
    return

  booking_cards = []
  for row in filtered_bookings:
    cancel_btn = widgets.Button(
      description="Cancel",
      button_style='danger',
      layout=widgets.Layout(
        width='100px',
        height='30px',
        margin='5px 0 0 0'
      )
    )

    cancel_btn.on_click(lambda btn, uid=row["UserId"], bdate=row["BookingDate"], btime=row["Time"]: bookings_cancel_booking(btn, uid, bdate, btime, return_to_callback))

    session_title = widgets.Label(value="📅 In-Store Session", layout=widgets.Layout(font_weight='bold'))
    user_id_label = widgets.Label(value=f"User: {row['UserId']}")
    date_label = widgets.Label(value=f"Date: {row['BookingDate']}")
    time_label = widgets.Label(value=f"Time: {row['Time']}")
    guests_label = widgets.Label(value=f"Guests: {row['NoGuests']}")

    button_container = widgets.HBox(
      [cancel_btn],
      layout=widgets.Layout(justify_content='center')
    )

    booking_card = widgets.VBox([
      session_title,
      user_id_label,
      date_label,
      time_label,
      guests_label,
      button_container
    ], layout=widgets.Layout(
      border='2px solid gray',
      padding='10px',
      margin='8px',
      width='380px'
    ))

    booking_cards.append(booking_card)

  # Grid layout for booking cards (2 columns max)
  bookings_grid = widgets.GridBox(
    booking_cards,
    layout=widgets.Layout(
      grid_template_columns="repeat(2, 400px)",
      justify_content='center',
      grid_gap='10px',
      margin='20px 0'
    )
  )

  booking_cards_container.children = [bookings_grid]


def create_bookings_view(book_callback, return_to_callback):
  # Remove expired bookings first
  bookings_remove_expired()

  # Build the bookings tab view
  book_button = widgets.Button(
    description="📅 Book New Session",
    button_style='success',
    layout=widgets.Layout(width='180px', height='35px')
  )

  book_button.on_click(lambda x: book_callback())

  # Load and sort bookings chronologically
  all_bookings = []
  if os.path.exists("Bookings.txt") and os.path.getsize("Bookings.txt") > 0:
    with open("Bookings.txt", 'r') as f:
      reader = csv.DictReader(f)
      for row in reader:
        all_bookings.append(row)

    # Sort by date, then by time
    all_bookings.sort(key=lambda x: (x["BookingDate"], x["Time"]))

  # Create date filter dropdown - OUTSIDE the info box
  unique_dates = sorted(list(set([b["BookingDate"] for b in all_bookings]))) if all_bookings else []
  date_options = ["All Dates"] + unique_dates

  date_filter = widgets.Dropdown(
    options=date_options,
    value="All Dates",
    description="Filter by Date:",
    style={'description_width': '95px'},
    layout=widgets.Layout(width='320px', margin='0 0 10px 0')
  )

  date_filter_container = widgets.HBox(
    [date_filter],
    layout=widgets.Layout(justify_content='center', margin='20px 0 10px 0')
  )

  # Single HTML info box with dark green background
  info_box_html = widgets.HTML(
    value="""
    <div style="
      background-color: #2E7D32;
      padding: 20px;
      border-radius: 8px;
      width: 750px;
      margin: 0 auto;
    ">
      <div style="
        display: flex;
        justify-content: space-between;
        align-items: center;
        margin-bottom: 20px;
      ">
        <span style="font-weight: bold; font-size: 16px; color: white;">
          📋 In-Store Session Bookings
        </span>
      </div>
      <div style="color: white; line-height: 1.8;">
        <div style="font-weight: bold;">Select a date to view capacity</div>
      </div>
    </div>
    """,
    layout=widgets.Layout(width='100%', margin='0 0 20px 0')
  )

  # Button positioned below the header
  button_container = widgets.HBox(
    [book_button],
    layout=widgets.Layout(justify_content='center', margin='20px 0')
  )

  # Container for booking cards (will be updated by filter)
  booking_cards_container = widgets.VBox()

  # Initial display (all bookings)
  bookings_filter_by_date(None, date_filter, booking_cards_container, info_box_html, all_bookings, return_to_callback)

  # Connect filter dropdown to update function
  date_filter.observe(
    lambda change: bookings_filter_by_date(change, date_filter, booking_cards_container, info_box_html, all_bookings, return_to_callback),
    names='value'
  )

  return widgets.VBox([
    date_filter_container,
    info_box_html,
    button_container,
    booking_cards_container
  ], layout=widgets.Layout(padding="20px", align_items='center'))

# Analytics Cell

**What This Cell Does:**

This cell creates an analytics dashboard for game performance tracking. It provides:
* **Charts View**: Bar charts showing "Most Rented Games" (with time period filtering: All Time, Last 180 Days, Last 30 Days) and "Games by Star Rating" for both video and board games
* **Underperformers View**: Identifies games that don't meet minimum rental or rating thresholds with adjustable sliders
* Two-panel layout with switchable views
* Visual indicators for underperforming games that may need removal
* Time period selector that appears/hides based on chart type selection

**How It Should Be Used:**
* Run this cell after the Helper Cell
* Creates a complete analytics interface accessible from the main dashboard
* Allows staff to view game performance metrics filtered by time period and identify underperforming inventory

---

## Functions:

**`analytics_create_chart(chart_type, time_period=None)`**
* Generates horizontal bar charts comparing video games and board games by rentals (with optional time filtering) or ratings
* **`chart_type`** - Either "Most Rented Games" or "Games by Star Rating"
* **`time_period`** - Optional number of days to filter rentals (30, 180, or None for all time)

**`analytics_update_charts(change, chart_selector, time_period_selector, chart_output)`**
* Updates the displayed chart when user changes the chart type or time period selection
* **`change`** - Event object from dropdown change
* **`chart_selector`** - Dropdown widget with chart type options
* **`time_period_selector`** - Dropdown widget with time period options (All Time, Last 180 Days, Last 30 Days)
* **`chart_output`** - Output widget where chart is displayed

**`analytics_ui(main_menu_callback)`**
* Creates and displays the complete analytics dashboard interface with both views, chart/time selectors, and all controls
* **`main_menu_callback`** - Function to call when "Main Menu" button is clicked

**`analytics_switch_view(view_index, current_view, charts_view_btn, underperformers_view_btn, content_area, charts_view, underperformers_view)`**
* Switches between Charts View and Underperformers View, updating button styles
* **`view_index`** - 0 for charts view, 1 for underperformers view
* **`current_view`** - List containing the current view index
* **`charts_view_btn`** - Button widget for charts view
* **`underperformers_view_btn`** - Button widget for underperformers view
* **`content_area`** - VBox widget that holds the active view
* **`charts_view`** - Widget containing the charts interface
* **`underperformers_view`** - Widget containing the underperformers interface

**`analytics_update_underperformers(change, min_rentals_slider, min_rating_slider, underperformers_output)`**
* Finds and displays games below the minimum rental count or rating thresholds
* **`change`** - Event object from slider change
* **`min_rentals_slider`** - IntSlider widget for minimum rental threshold
* **`min_rating_slider`** - FloatSlider widget for minimum rating threshold
* **`underperformers_output`** - Output widget where underperforming games are displayed

In [25]:
# ============================================
# ANALYTICS & INVENTORY PRUNING
# ============================================

def analytics_create_chart(chart_type, time_period=None):

  board_games = load_games("Board_Game_Info.txt", "board")
  video_games = load_games("Video_Game_Info.txt", "video")

  if chart_type == "Rental Distribution":
    # Calculate total rentals for video games vs board games
    video_rentals = sum([get_game_rent_count_by_period(g['GameID'], time_period) for g in video_games])
    board_rentals = sum([get_game_rent_count_by_period(g['GameID'], time_period) for g in board_games])

    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    sizes = [video_rentals, board_rentals]
    labels = [f'Video Games\n({video_rentals} rentals)', f'Board Games\n({board_rentals} rentals)']
    colors = ['#4CAF50', '#FF9800']
    explode = (0.05, 0.05)

    chart_title = 'Game Rental Distribution'
    if time_period == 30:
      chart_title += ' (Last 30 Days)'
    elif time_period == 180:
      chart_title += ' (Last 180 Days)'
    else:
      chart_title += ' (All Time)'

    wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
                                       startangle=90, explode=explode, textprops={'fontsize': 12, 'weight': 'bold'})

    ax.set_title(chart_title, fontsize=16, fontweight='bold', pad=20)

    # Make percentage text larger and white
    for autotext in autotexts:
      autotext.set_color('white')
      autotext.set_fontsize(14)
      autotext.set_weight('bold')

    plt.tight_layout()
    return

  elif chart_type == "Most Rented Games":
    # Update rental counts based on time period
    for game in board_games:
      game['RentalCount'] = get_game_rent_count_by_period(game['GameID'], time_period)
    for game in video_games:
      game['RentalCount'] = get_game_rent_count_by_period(game['GameID'], time_period)

    metric_key = 'RentalCount'
    chart_title = 'Most Rented Games'
    if time_period == 30:
      chart_title += ' (Last 30 Days)'
    elif time_period == 180:
      chart_title += ' (Last 180 Days)'
    else:
      chart_title += ' (All Time)'

    xlabel = 'Number of Rentals'
    vg_color = '#4CAF50'
    bg_color = '#FF9800'
    value_format = '{:.0f}'
    x_limit = None

  else:  # Games by Star Rating
    metric_key = 'Rating'
    chart_title = 'Games by Star Rating'
    xlabel = 'Average Rating (out of 5)'
    vg_color = '#2196F3'
    bg_color = '#9C27B0'
    value_format = '{:.1f}'
    x_limit = (0, 5)

  fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
  fig.patch.set_alpha(0)
  ax1.patch.set_alpha(0)
  ax2.patch.set_alpha(0)

  fig.suptitle(chart_title, fontsize=16, fontweight='bold')

  vg_names = [g['Name'][:20] + '...' if len(g['Name']) > 20 else g['Name'] for g in video_games]
  vg_values = [g[metric_key] if g[metric_key] is not None else 0 for g in video_games]

  ax1.barh(vg_names, vg_values, color=vg_color)
  ax1.set_xlabel(xlabel, fontweight='bold')
  ax1.set_title('Video Games', fontsize=14, fontweight='bold')
  ax1.invert_yaxis()

  if x_limit:
    ax1.set_xlim(x_limit)

  # Add value labels
  for i, v in enumerate(vg_values):
    offset = 0.05 if x_limit else v * 0.02
    ax1.text(v + offset, i, value_format.format(v), va='center')

  bg_names = [g['Name'][:20] + '...' if len(g['Name']) > 20 else g['Name'] for g in board_games]
  bg_values = [g[metric_key] if g[metric_key] is not None else 0 for g in board_games]

  ax2.barh(bg_names, bg_values, color=bg_color)
  ax2.set_xlabel(xlabel, fontweight='bold')
  ax2.set_title('Board Games', fontsize=14, fontweight='bold')
  ax2.invert_yaxis()

  if x_limit:
    ax2.set_xlim(x_limit)

  # Add value labels
  for i, v in enumerate(bg_values):
    offset = 0.05 if x_limit else v * 0.02
    ax2.text(v + offset, i, value_format.format(v), va='center')

  plt.tight_layout()


def analytics_update_charts(change, chart_selector, time_period_selector, chart_output):
  selected = chart_selector.value

  # Get time period (only relevant for rental charts)
  time_period = None
  if selected == "Most Rented Games" or selected == "Rental Distribution":
    period_value = time_period_selector.value
    if period_value == "Last 30 Days":
      time_period = 30
    elif period_value == "Last 180 Days":
      time_period = 180
    # if NO SELECTION, assumes all time (hence None)

  # Clear previous output
  chart_output.clear_output(wait=True)
  with chart_output:
    # Show the chart
    analytics_create_chart(selected, time_period)
    plt.show()


def analytics_ui(main_menu_callback):

  # View selection buttons
  # -----------------------

  charts_view_btn = widgets.Button(
    description="📊 Charts View",
    layout=widgets.Layout(width='50%', height='50px'),
    style={'font_weight': 'bold'}
  )

  underperformers_view_btn = widgets.Button(
    description="⚠️ Underperformers",
    layout=widgets.Layout(width='50%', height='50px'),
    style={'font_weight': 'bold'}
  )

  view_buttons = widgets.HBox(
    [charts_view_btn, underperformers_view_btn],
    layout=widgets.Layout(width='95%', margin='0 auto 20px auto')
  )

  # Charts View Components
  # ----------------------

  chart_selector = widgets.Dropdown(
    options=['Most Rented Games', 'Games by Star Rating', 'Rental Distribution'],
    description='View:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
  )

  # Time period selector for rental charts
  time_period_selector = widgets.Dropdown(
    options=['All Time', 'Last 180 Days', 'Last 30 Days'],
    value='All Time',
    description='Period:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
  )

  chart_output = widgets.Output(
    layout=widgets.Layout(
      width='100%',
      min_height='600px'
    )
  )

  selector_container = widgets.HBox(
    [chart_selector, time_period_selector],
    layout=widgets.Layout(justify_content='center', margin='20px 0',
                          grid_gap='20px')
  )

  charts_view = widgets.VBox([
    selector_container,
    chart_output
  ])

  # Underperformers View Components
  # --------------------------------

  min_rentals_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=20,
    step=1,
    description='Min Rentals:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
  )

  min_rating_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=5.0,
    step=0.5,
    description='Min Rating:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
  )

  sliders_container = widgets.VBox(
    [min_rentals_slider, min_rating_slider],
    layout=widgets.Layout(
      align_items='center',
      margin='20px 0'
    )
  )

  underperformers_output = widgets.Output(
    layout=widgets.Layout(
      width='100%',
      min_height='600px',
      max_height='600px',
      overflow_y='auto'
    )
  )

  underperformers_view = widgets.VBox([
    sliders_container,
    underperformers_output
  ])

  # Content area
  # ------------

  content_area = widgets.VBox(
    layout=widgets.Layout(
      min_height='600px',
      border='2px solid gray',
      padding='20px'
    )
  )

  # View switching logic
  # --------------------

  current_view = [0]  # 0=charts, 1=underperformers

  def switch_and_update_time_selector(view_idx):
    analytics_switch_view(
      view_idx,
      current_view,
      charts_view_btn,
      underperformers_view_btn,
      content_area,
      charts_view,
      underperformers_view
    )
    # Show/hide time period selector based on chart type
    update_time_period_visibility()

  charts_view_btn.on_click(lambda btn: switch_and_update_time_selector(0))
  underperformers_view_btn.on_click(lambda btn: switch_and_update_time_selector(1))

  # Function to show/hide time period selector
  def update_time_period_visibility(change=None):
    if chart_selector.value == "Most Rented Games" or chart_selector.value == "Rental Distribution":
      time_period_selector.layout.visibility = 'visible'
    else:
      time_period_selector.layout.visibility = 'hidden'

  # Initial loads
  # -------------

  analytics_update_charts(None, chart_selector, time_period_selector, chart_output)
  analytics_update_underperformers(
    None,
    min_rentals_slider,
    min_rating_slider,
    underperformers_output
  )

  # Update when sliders change
  min_rentals_slider.observe(
    lambda change: analytics_update_underperformers(
      change,
      min_rentals_slider,
      min_rating_slider,
      underperformers_output
    ),
    names='value'
  )

  min_rating_slider.observe(
    lambda change: analytics_update_underperformers(
      change,
      min_rentals_slider,
      min_rating_slider,
      underperformers_output
    ),
    names='value'
  )

  # Update when dropdown changes
  chart_selector.observe(
    lambda change: [
      analytics_update_charts(change, chart_selector, time_period_selector, chart_output),
      update_time_period_visibility(change)
    ],
    names='value'
  )

  # Update when time period changes
  time_period_selector.observe(
    lambda change: analytics_update_charts(
      change,
      chart_selector,
      time_period_selector,
      chart_output
    ),
    names='value'
  )

  # Set initial visibility
  update_time_period_visibility()

  title = widgets.Label(
    value="📊 Analytics Dashboard",
    layout=widgets.Layout(margin='10px 0')
  )

  clear_output(wait=True)
  # Header
  header = widgets.HTML("<h2 style='color:#0b8517; margin:0'>PIXEL VAULT – ANALYTICS</h2>")

  main_menu_button = widgets.Button(
    description="🏠 Main Menu",
    button_style='success',
    layout=widgets.Layout(width='150px', height='40px'),
    style={'font_weight': 'bold'}
  )

  main_menu_button.on_click(lambda btn: main_menu_callback())

  header_frame = widgets.HBox(
    [header, main_menu_button],
    layout=widgets.Layout(
      width='95%',
      margin='0 auto 20px auto',
      padding='15px 20px',
      border='2px solid #0b8517',
      justify_content='space-between',
      align_items='center'
    )
  )

  # Content
  content = widgets.VBox([
    title,
    view_buttons,
    content_area
  ], layout=widgets.Layout(padding="20px", align_items='center'))

  main_container = widgets.VBox(
    [header_frame, content],
    layout=widgets.Layout(
      width='90%',
      margin='0 auto',
      padding='20px'
    )
  )

  # Set initial view
  analytics_switch_view(
    0,
    current_view,
    charts_view_btn,
    underperformers_view_btn,
    content_area,
    charts_view,
    underperformers_view
  )

  display(main_container)


def analytics_switch_view(
  view_index,
  current_view,
  charts_view_btn,
  underperformers_view_btn,
  content_area,
  charts_view,
  underperformers_view
):
  current_view[0] = view_index

  if view_index == 0:
    charts_view_btn.button_style = 'success'
    underperformers_view_btn.button_style = ''
    content_area.children = [charts_view]
  else:
    charts_view_btn.button_style = ''
    underperformers_view_btn.button_style = 'success'
    content_area.children = [underperformers_view]


def analytics_update_underperformers(
  change,
  min_rentals_slider,
  min_rating_slider,
  underperformers_output
):
  min_rentals = min_rentals_slider.value
  min_rating = min_rating_slider.value

  board_games = load_games("Board_Game_Info.txt", "board")
  video_games = load_games("Video_Game_Info.txt", "video")
  all_games = board_games + video_games

  underperforming = []

  for game in all_games:
    rental_count = game['RentalCount'] if game['RentalCount'] is not None else 0
    rating = game['Rating'] if game['Rating'] is not None else 0

    if rental_count < min_rentals or rating < min_rating:
      underperforming.append({
        'name': game['Name'],
        'type': game['Type'],
        'rentals': rental_count,
        'rating': rating
      })

  underperformers_output.clear_output(wait=True)
  with underperformers_output:
    if not underperforming:
      display(widgets.HTML(
        "<div style='text-align: center; margin: 50px; color: green; font-size: 18px;'>"
        "✔ All games meet the performance criteria!"
        "</div>"
      ))
    else:
      display(widgets.HTML(
        "<div style='text-align: center; margin: 20px 0; color: #FF5722; font-size: 16px; font-weight: bold;'>"
        f"{len(underperforming)} game(s) are underperforming and may need to be removed"
        "</div>"
      ))

      game_cards = []
      for game in underperforming:
        game_type_emoji = "🎮" if game['type'] == 'video' else "🎲"
        stars = create_star_display(game['rating'], "string") if game['rating'] > 0 else "No ratings"

        game_card = widgets.HTML(
          f"<div style='border: 1px solid #000; padding: 15px; margin: 10px; "
          f"background-color: #ffffff; color: #000000; border-radius: 5px;'>"
          f"<strong>{game_type_emoji} {game['name']}</strong><br>"
          f"Rentals: {game['rentals']} (Min: {min_rentals})<br>"
          f"Rating: {stars} ({game['rating']:.1f}/5.0, Min: {min_rating:.1f})"
          f"</div>"
        )
        game_cards.append(game_card)

      # Display games side by side in a grid
      games_grid = widgets.GridBox(
        game_cards,
        layout=widgets.Layout(
          grid_template_columns="repeat(2, 1fr)",
          grid_gap='10px',
          width='100%'
        )
      )
      display(games_grid)

# Search Cell

**What This Cell Does:**

This cell provides real-time search functionality across all games (board and video). It:
* Searches by game title, genre, or platform (for video games)
* Updates results instantly as the user types
* Displays matching games in a grid with full game cards (image, info, buttons)
* Shows all games when search box is empty
* Displays "No games found" message if no matches

**How It Should Be Used:**
* Run this cell after the Helper Cell and Game Card Display Cell
* Creates the search interface that appears as a tab in the main dashboard
* Provides live filtering of games without needing to press a search button

---

## Functions:

**`search_filter_games(search_term, all_games)`**
* Filters games based on search term, matching against title, genre, or platform (video games only)
* **`search_term`** - Text to search for
* **`all_games`** - List of all game dictionaries to search through

**`search_update_results(change, search_input, results_container, all_games, main_menu_callback)`**
* Updates the displayed game cards whenever the search input changes
* **`change`** - Event object from text input change
* **`search_input`** - Text widget containing the search query
* **`results_container`** - VBox widget where filtered game cards are displayed
* **`all_games`** - Same as above
* **`main_menu_callback`** - Function to call to return to main menu

**`search_ui(main_menu_callback)`**
* Creates and displays the complete search interface with search bar and results container

In [26]:
# ============================================
# SEARCH
# ============================================

# Real-time search across all games by title, genre, or platform (for video games)

def search_filter_games(search_term, all_games):
  # Return all games if search is empty
  if not search_term.strip():
    return all_games

  search_term = search_term.lower()
  filtered = []

  for game in all_games:
    # Match title
    if search_term in game['Name'].lower():
      filtered.append(game)
      continue

    # Match genre
    if search_term in game['Genre'].lower():
      filtered.append(game)
      continue

    # Match platform (video games only, stored in NoPlayers field)
    if game['Type'] == 'video' and 'NoPlayers' in game:
      if search_term in game['NoPlayers'].lower():
        filtered.append(game)

  return filtered


def search_update_results(change, search_input, results_container, all_games, main_menu_callback):
  # Called whenever search input changes
  search_term = search_input.value
  filtered_games = search_filter_games(search_term, all_games)

  if not filtered_games:
    results_container.children = [
      widgets.Label(
        value=f"No games found for '{search_term}'",
        layout=widgets.Layout(margin='50px auto')
      )
    ]
  else:
    # Create game cards with callbacks
    cards = [create_game_card(
      g,
      lambda game: rent_game(game, lambda: search_ui(main_menu_callback), lambda: search_ui(main_menu_callback)),
      lambda game: return_game(game, lambda: search_ui(main_menu_callback), lambda: search_ui(main_menu_callback)),
      lambda game: show_reviews(game, lambda: search_ui(main_menu_callback), lambda: search_ui(main_menu_callback))
    ) for g in filtered_games]

    grid = widgets.GridBox(
      cards,
      layout=widgets.Layout(
        grid_template_columns="repeat(3, 0fr)",
        justify_content="center",
        justify_items="center",
        align_items="flex-start",
        grid_gap="5px",
        width="100%"
      )
    )
    results_container.children = [grid]


def search_ui(main_menu_callback):

  # Load all games (board + video)
  board_games = load_games("Board_Game_Info.txt", "board")
  video_games = load_games("Video_Game_Info.txt", "video")
  all_games = board_games + video_games

  # Mark rented games and attach rental info
  rentals = load_rentals()

  for game in all_games:
    if game["GameID"] in rentals:
      game["Available"] = False
      game["RentalInfo"] = rentals[game["GameID"]]

  # Search input
  search_input = widgets.Text(
    placeholder='Search by title, genre, or platform (e.g., Xbox, PlayStation, Nintendo)...',
    layout=widgets.Layout(width='90%', height='50px'),
    style={'description_width': 'initial'}
  )

  # Container for results
  results_container = widgets.VBox(
    layout=widgets.Layout(
      min_height='600px',
      border='2px solid gray',
      padding='20px'
    )
  )

  # Real-time search as user types
  search_input.observe(lambda change: search_update_results(change, search_input, results_container, all_games, main_menu_callback), names='value')

  # Show all games initially
  search_update_results(None, search_input, results_container, all_games, main_menu_callback)

  return widgets.VBox([search_input, results_container])

# Dashboard Cell

**What This Cell Does:**

This cell creates the main dashboard interface for the game rental system. It:
* Displays a tabbed interface with four sections: Video Games, Board Games, Bookings, and Search
* Shows game cards in a grid layout for each game type
* Handles navigation between tabs with highlighted active tab
* Provides access to the Analytics dashboard via a button in the header
* Manages game availability state (marks rented games as unavailable)
* Refreshes game data when returning from rental/return operations

**How It Should Be Used:**
* Run this cell after all previous cells are loaded
* Call the `dashboard()` function to display the main interface
* This is the main entry point that users interact with
* All other interfaces (rent, return, reviews, bookings, analytics) return here

---

**Important Information**

During the creation of the dashboard there was an attempt to create a grid that would dynamically adapt to the current size of the screen, however this caused what appeared to be an "overflow" in the cache of colab, causing the analytics segment to completely disregard the logic of the program. This means that the current dashboard is set to fixed sizes so that this error is not caused. The game cards have been made smaller when colab is fully open on the tab, but this is to accommodate still being able to view and use the cards when text files are open or the colab tab is not in full screen. During an actual deployment of this python code the grid creation would be:

* `grid = widgets.GridBox(
    cards,
    layout=widgets.Layout(
        grid_template_columns="repeat(auto-fit, minmax(280px, 320px))",
        justify_content="center",
        justify_items="center",
        align_items="flex-start",
        grid_gap="10px",
        width="100%",
        padding='10px'
    )
)`

Along with other code to then dynamically adapt to the screen size.

---

## Functions:

**`dashboard_redisplay(main_container)`**
* Quickly redisplays the dashboard without reloading data (for minor updates)
* **`main_container`** - VBox widget containing the entire dashboard

**`dashboard_make_grid(games, main_container)`**
* Creates a grid layout of game cards with rent/return/review buttons
* **`games`** - List of game dictionaries to display
* **`main_container`** - Same as above

**`dashboard_refresh_cards(board_games, video_games, rentals, video_box, board_box, bookings_box, search_box, main_container)`**
* Reloads all game data from CSV files and rebuilds all tab content (used after rentals/returns)
* **`board_games`** - List to store board game data
* **`video_games`** - List to store video game data
* **`rentals`** - Dictionary to store active rental records
* **`video_box`** - VBox widget for video games tab content
* **`board_box`** - VBox widget for board games tab content
* **`bookings_box`** - VBox widget for bookings tab content
* **`search_box`** - VBox widget for search tab content
* **`main_container`** - Same as above

**`dashboard_update_buttons(current_tab, video_btn, board_btn, bookings_btn, search_btn, content_area, video_box, board_box, bookings_box, search_box)`**
* Updates button styles to highlight the active tab and displays the corresponding content
* **`current_tab`** - List containing current tab index (0=video, 1=board, 2=bookings, 3=search)
* **`video_btn`** - Button widget for video games tab
* **`board_btn`** - Button widget for board games tab
* **`bookings_btn`** - Button widget for bookings tab
* **`search_btn`** - Button widget for search tab
* **`content_area`** - VBox widget where tab content is displayed
* **`video_box`** - Same as above
* **`board_box`** - Same as above
* **`bookings_box`** - Same as above
* **`search_box`** - Same as above

**`dashboard_on_video_click(btn, current_tab, video_btn, board_btn, bookings_btn, search_btn, content_area, video_box, board_box, bookings_box, search_box)`**
* Switches to the video games tab when button is clicked
* **`btn`** - Same as above
* **`current_tab`** - Same as above
* **`video_btn`** - Same as above
* **`board_btn`** - Same as above
* **`bookings_btn`** - Same as above
* **`search_btn`** - Same as above
* **`content_area`** - Same as above
* **`video_box`** - Same as above
* **`board_box`** - Same as above
* **`bookings_box`** - Same as above
* **`search_box`** - Same as above

**`dashboard_on_board_click(...)`**
* Switches to the board games tab when button is clicked
* **Parameters:** Same as `dashboard_on_video_click`

**`dashboard_on_bookings_click(...)`**
* Switches to the bookings tab when button is clicked
* **Parameters:** Same as `dashboard_on_video_click`

**`dashboard_on_search_click(...)`**
* Switches to the search tab when button is clicked
* **Parameters:** Same as `dashboard_on_video_click`

**`dashboard()`**
* Creates and displays the complete dashboard interface with all tabs, buttons, and initial content
* **No parameters**

In [27]:
# ============================================
# DASHBOARD
# ============================================

# Main menu with tabs for video games, board games, and bookings. Handles game state and navigation

def dashboard_redisplay(main_container):
  # Quick redisplay without reload
  clear_output(wait=True)
  display(main_container)


def dashboard_make_grid(games, main_container):
  # Create grid of game cards
  cards = [create_game_card(
    g,
    lambda game: rent_game(
      game,
      dashboard,
      lambda: dashboard_redisplay(main_container)
    ),
    lambda game: return_game(
      game,
      dashboard,
      lambda: dashboard_redisplay(main_container)
    ),
    lambda game: show_reviews(
      game,
      dashboard,
      lambda: dashboard_redisplay(main_container)
    )
  ) for g in games]

  grid = widgets.GridBox(
    cards,
    layout=widgets.Layout(
      grid_template_columns="repeat(3, 0fr)",
      justify_content="center",
      justify_items="center",
      align_items="flex-start",
      grid_gap="5px",
      width="100%"
    )
  )
  return grid


def dashboard_refresh_cards(
  board_games,
  video_games,
  rentals,
  video_box,
  board_box,
  bookings_box,
  search_box,
  main_container
):
  # Reload game data and rebuild cards
  board_games[:] = load_games("Board_Game_Info.txt", "board")
  video_games[:] = load_games("Video_Game_Info.txt", "video")

  rentals.clear()
  rentals.update(load_rentals())

  for game in board_games + video_games:
    if game["GameID"] in rentals:
      game["Available"] = False
      game["RentalInfo"] = rentals[game["GameID"]]

  video_box.children = [dashboard_make_grid(video_games, main_container)]
  board_box.children = [dashboard_make_grid(board_games, main_container)]
  bookings_box.children = [create_bookings_view(
    lambda: book_session(
      dashboard,
      lambda: dashboard_redisplay(main_container)
    ),
    dashboard
  )]
  search_box.children = [search_ui(dashboard)]


def dashboard_update_buttons(
  current_tab,
  video_btn,
  board_btn,
  bookings_btn,
  search_btn,
  content_area,
  video_box,
  board_box,
  bookings_box,
  search_box
):
  # Highlight active tab and show its content
  if current_tab[0] == 0:
    video_btn.button_style = 'success'
    board_btn.button_style = ''
    bookings_btn.button_style = ''
    search_btn.button_style = ''
    content_area.children = video_box.children
  elif current_tab[0] == 1:
    video_btn.button_style = ''
    board_btn.button_style = 'success'
    bookings_btn.button_style = ''
    search_btn.button_style = ''
    content_area.children = board_box.children
  elif current_tab[0] == 2:
    video_btn.button_style = ''
    board_btn.button_style = ''
    bookings_btn.button_style = 'success'
    search_btn.button_style = ''
    content_area.children = bookings_box.children
  else:
    video_btn.button_style = ''
    board_btn.button_style = ''
    bookings_btn.button_style = ''
    search_btn.button_style = 'success'
    content_area.children = search_box.children


def dashboard_on_video_click(
  btn,
  current_tab,
  video_btn,
  board_btn,
  bookings_btn,
  search_btn,
  content_area,
  video_box,
  board_box,
  bookings_box,
  search_box
):
  current_tab[0] = 0
  dashboard_update_buttons(
    current_tab,
    video_btn,
    board_btn,
    bookings_btn,
    search_btn,
    content_area,
    video_box,
    board_box,
    bookings_box,
    search_box
  )


def dashboard_on_board_click(
  btn,
  current_tab,
  video_btn,
  board_btn,
  bookings_btn,
  search_btn,
  content_area,
  video_box,
  board_box,
  bookings_box,
  search_box
):
  current_tab[0] = 1
  dashboard_update_buttons(
    current_tab,
    video_btn,
    board_btn,
    bookings_btn,
    search_btn,
    content_area,
    video_box,
    board_box,
    bookings_box,
    search_box
  )


def dashboard_on_bookings_click(
  btn,
  current_tab,
  video_btn,
  board_btn,
  bookings_btn,
  search_btn,
  content_area,
  video_box,
  board_box,
  bookings_box,
  search_box
):
  current_tab[0] = 2
  dashboard_update_buttons(
    current_tab,
    video_btn,
    board_btn,
    bookings_btn,
    search_btn,
    content_area,
    video_box,
    board_box,
    bookings_box,
    search_box
  )


def dashboard_on_search_click(
  btn,
  current_tab,
  video_btn,
  board_btn,
  bookings_btn,
  search_btn,
  content_area,
  video_box,
  board_box,
  bookings_box,
  search_box
):
  current_tab[0] = 3
  dashboard_update_buttons(
    current_tab,
    video_btn,
    board_btn,
    bookings_btn,
    search_btn,
    content_area,
    video_box,
    board_box,
    bookings_box,
    search_box
  )


def dashboard():
  clear_output(wait=True)
  anchor = widgets.HTML("<div id='top-anchor'></div>")
  display(anchor)

  # Load games from CSV files
  board_games = load_games("Board_Game_Info.txt", "board")
  video_games = load_games("Video_Game_Info.txt", "video")

  # Mark rented games
  rentals = load_rentals()

  for game in board_games + video_games:
    if game["GameID"] in rentals:
      game["Available"] = False
      game["RentalInfo"] = rentals[game["GameID"]]

  # Tab content containers
  video_box = widgets.VBox()
  board_box = widgets.VBox()
  bookings_box = widgets.VBox()
  search_box = widgets.VBox()

  # Tab navigation
  current_tab = [0]  # 0=video, 1=board, 2=bookings, 3=search

  video_btn = widgets.Button(
    description="🎮 Video Games",
    layout=widgets.Layout(width='25%', height='50px'),
    style={'font_weight': 'bold'}
  )
  board_btn = widgets.Button(
    description="🎲 Board Games",
    layout=widgets.Layout(width='25%', height='50px'),
    style={'font_weight': 'bold'}
  )
  bookings_btn = widgets.Button(
    description="📋 Bookings",
    layout=widgets.Layout(width='25%', height='50px'),
    style={'font_weight': 'bold'}
  )
  search_btn = widgets.Button(
    description="🔍 Search",
    layout=widgets.Layout(width='25%', height='50px'),
    style={'font_weight' : 'bold'}
  )

  content_area = widgets.VBox(
    layout=widgets.Layout(
      min_height='600px',
      border='2px solid gray',
      padding='20px'
    )
  )

  button_bar = widgets.HBox(
    [video_btn, board_btn, bookings_btn, search_btn],
    layout=widgets.Layout(width='95%', margin='0 auto')
  )

  tabs_container = widgets.VBox(
    [button_bar, content_area],
    layout=widgets.Layout(width='95%', margin='20px auto')
  )

  # Header with search button
  header = widgets.HTML("<h2 style='color:#0b8517; margin:0'>PIXEL VAULT – HOME</h2>")

  analytics_button = widgets.Button(
    description="📊 Analytics",
    button_style='success',
    layout=widgets.Layout(width='150px', height='40px'),
    style={'font_weight': 'bold'}
  )

  analytics_button.on_click(lambda btn: analytics_ui(dashboard))

  header_frame = widgets.HBox(
    [header, analytics_button],
    layout=widgets.Layout(
      width='95%',
      margin='0 auto 20px auto',
      padding='15px 20px',
      border='2px solid #0b8517',
      justify_content='space-between',
      align_items='center'
    )
  )

  main_container = widgets.VBox(
    [header_frame, tabs_container],
    layout=widgets.Layout(
      width='90%',
      margin='0 auto',
      padding='20px'
    )
  )

  dashboard_refresh_cards(
    board_games,
    video_games,
    rentals,
    video_box,
    board_box,
    bookings_box,
    search_box,
    main_container
  )

  video_btn.on_click(
    lambda btn: dashboard_on_video_click(
      btn,
      current_tab,
      video_btn,
      board_btn,
      bookings_btn,
      search_btn,
      content_area,
      video_box,
      board_box,
      bookings_box,
      search_box
    )
  )
  board_btn.on_click(
    lambda btn: dashboard_on_board_click(
      btn,
      current_tab,
      video_btn,
      board_btn,
      bookings_btn,
      search_btn,
      content_area,
      video_box,
      board_box,
      bookings_box,
      search_box
    )
  )
  bookings_btn.on_click(
    lambda btn: dashboard_on_bookings_click(
      btn,
      current_tab,
      video_btn,
      board_btn,
      bookings_btn,
      search_btn,
      content_area,
      video_box,
      board_box,
      bookings_box,
      search_box
    )
  )
  search_btn.on_click(
    lambda btn: dashboard_on_search_click(
      btn,
      current_tab,
      video_btn,
      board_btn,
      bookings_btn,
      search_btn,
      content_area,
      video_box,
      board_box,
      bookings_box,
      search_box
    )
  )

  dashboard_update_buttons(
    current_tab,
    video_btn,
    board_btn,
    bookings_btn,
    search_btn,
    content_area,
    video_box,
    board_box,
    bookings_box,
    search_box
  )

  display(anchor)
  display(main_container)

## Security Issues
**1. Weak Authentication**  
* User credentials are stored and checked without encryption, making them easy to read or steal if accessed.

**2. Poor Access Control**  
* There is no check to confirm whether the user accessing the database is an administrator, allowing unauthorised actions.

**3. Data Integrity Risks**  
* The system does not properly validate or protect data, meaning records can be accidentally changed or duplicated.

**4. No Error Handling**  
* The program does not handle errors safely, which may cause crashes or expose internal system details.

**5. No Logging or Auditing**  
* There is no record of user activity, making it impossible to track who accessed or modified data.


In [28]:
# ============================================
# START
# ============================================

dashboard()

HTML(value="<div id='top-anchor'></div>")

HTML(value="<div id='top-anchor'></div>")